In [ ]:
from huggingface_hub import login
login() 

In [ ]:
# Configuration of carpetas for entorno LOCAL
from pathlib import Path
BASE_DIR = Path.cwd().parent
BASE_DIR.mkdir(exist_ok=True)

for sub in ["context", "models", "processed_data"]:
    (BASE_DIR / sub).mkdir(parents=True, exist_ok=True)

print("Base:", BASE_DIR.resolve())
print("Structure created (if not existed):")
for p in ["context", "models", "proccesed data"]:
    print(" -", (BASE_DIR / p).resolve())

# Score: the dataset must existir in: CORPUS/proccesed data/wuxia_zh_en_clean


In [ ]:
from dataclasses import dataclass

@dataclass
class Config:
    # Paths (local)
    dataset_dir: Path  = BASE_DIR / "processed_data" / "wuxia_zh_en_clean"  
    output_dir: Path   = BASE_DIR / "models" / "qwen3"
       
    evaluation_dir: Path = BASE_DIR / "evaluation"
    translate_dir: Path = BASE_DIR / "evaluation" / "translate" / "llm"
    translate_file: Path =   "qwen3.txt"
    results_file: Path = "llm_results.txt"
    

    # Model
    # model_ckpt: str = "Qwen/Qwen-7B-Chat"
    model_ckpt: str= "Qwen/Qwen3-4B-Instruct-2507"

cfg = Config()
print(cfg)


In [ ]:
import random, numpy as np, torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [ ]:
from datasets import load_from_disk


raw_ds = load_from_disk(cfg.dataset_dir)
print(raw_ds)

In [ ]:
example = raw_ds["train"][0]
print("ZH:", example["zh"])
print("EN:", example["en"])

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig

# Cargamos the tokenizador
tokenizer = AutoTokenizer.from_pretrained(cfg.model_ckpt, trust_remote_code=True)

# Cargamos the model in mean precision and asignamos automatically to the GPU available
model = AutoModelForCausalLM.from_pretrained(
    cfg.model_ckpt, 
    device_map="auto",       
    dtype=torch.float16  , 
    trust_remote_code=True
)

# (Optional) Verificamos if is has loaded in GPU
device = model.device
print("Model loaded in device:", device)

In [ ]:
# Calculate number of parameters (optional, can take a slightly)
total_params = sum(p.numel() for p in model.parameters())
print(f"Parameters total of the model: {total_params:,}")

In [ ]:
# Ensure pad/eos
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
model.generation_config = GenerationConfig.from_model_config(model.config)
model.generation_config.pad_token_id = tokenizer.pad_token_id
model.generation_config.eos_token_id = tokenizer.eos_token_id
model.generation_config.use_cache = False  # evita the bug of past_key_values

In [ ]:
from transformers.generation import StoppingCriteria, StoppingCriteriaList

class StopOnSubsequence(StoppingCriteria):
    def __init__(self, stop_ids):
        super().__init__()
        # not asumas device yet
        self.stop_ids = torch.tensor(stop_ids, dtype=torch.long)

    def __call__(self, input_ids: torch.LongTensor, scores: torch.FloatTensor, **kwargs) -> bool:
        seq = input_ids[0]
        L = self.stop_ids.numel()
        
        stop_ids = self.stop_ids.to(seq.device)
        if seq.numel() >= L and torch.equal(seq[-L:], stop_ids):
            return True
        return False



# --- Construction of prompt ChatML (without templates) ---
def _build_chatml_prompt(system_msg: str, user_payload: str) -> str:
    # Format oficial of Qwen-Chat:
    # <|im_start|>system\n{...}<|im_end|>\n<|im_start|>user\n{...}<|im_end|>\n<|im_start|>assistant\n
    return (
        "<|im_start|>system\n"
        + system_msg
        + "<|im_end|>\n"
        + "<|im_start|>user\n"
        + user_payload
        + "<|im_end|>\n"
        + "<|im_start|>assistant\n"
    )


# --- Traductor robusto for Qwen-7B-Chat (compatible with tu block) ---
@torch.no_grad()
def translate_with_prompt_qwen(
                                chinese_text,
                                shots=None,
                                system_msg=None,
                                user_instruction=None,
                                max_new_tokens=256,
                            ):
    # if system_msg is None:
    #     system_msg = (
    #         "You are a professional translator specializing in Chinese wuxia literature. "
    #         "Translate the text into accurate, natural English while keeping all personal names, "
    #         "techniques, and cultural terms in pinyin. Do not add or omit any information."
    #     )
    # if user_instruction is None:
    #     user_instruction = "Translate the following Chinese text into faithful English:"

    # # Build payload of the user (with or without few-shot)
    # if shots:
    #     lines = ["Here are some examples of translations:\n"]
    #     for i, ex in enumerate(shots, 1):
    #         lines.append(f"Example {i}\nChinese: {ex['zh']}\nEnglish: {ex['en']}\n")
    #     lines.append(f"Now translate the following text:\nChinese: {chinese_text}\nEnglish:")
    #     user_payload = "\n".join(lines)
    # else:
    #     user_payload = f"{user_instruction}\nChinese: {chinese_text}\nEnglish:"

    # prompt = _build_chatml_prompt(system_msg, user_payload)


    # =========================
    if system_msg is None:
        system_msg = (
            "You are a professional translator specializing in Chinese wuxia literature. "
            "Translate the text into accurate, natural English while keeping all personal names, "
            "techniques, and cultural terms in pinyin. Do not add or omit any information."
        )

    if user_instruction is None:
        user_instruction = (
            "Translate the following Chinese text into faithful, idiomatic English prose:"
        )

    # =========================
    # 2. Construction of the payload few-shot
    # =========================
    if shots:
        examples = "\n".join(
            f"Example {i}\nChinese: {ex['zh']}\nEnglish: {ex['en']}\n"
            for i, ex in enumerate(shots, 1)
        )
        system_msg = (
            f"{system_msg}\n\n"
            f"Below are example translations to guide your style:\n"
            f"{examples}\n"
            "Use these examples for reference, but DO NOT repeat them or mention them in your output."
        )

    # =========================
    # 3. Construction of the prompt ChatML
    # =========================
    # ChatML standard: <|im_start|>system ... <|im_end|> <|im_start|>user ... <|im_end|> <|im_start|>assistant
    prompt = (
        f"<|im_start|>system\n{system_msg}\n<|im_end|>\n"
        f"<|im_start|>user\n"
        f"{user_instruction}\n"
        f"Convert all Chinese personal names, place names, and martial arts terms into pinyin romanization (do not translate them into English). Respond ONLY with the English translation:\n"
        f"Chinese: {chinese_text}\nEnglish:"
        f"<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )
    
    enc = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(device)
    input_ids = enc["input_ids"]
    attention_mask = enc.get("attention_mask", torch.ones_like(input_ids))

    # Stop to the produce <|im_end|>
    stop_ids = tokenizer.encode("<|im_end|>", add_special_tokens=False)
    stopping = StoppingCriteriaList([StopOnSubsequence(stop_ids)])

    # Generation determinista and corta
    gen_ids = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        temperature=0.0,
        top_p=1.0,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,  # can that no is use; the stop it garantiza
        use_cache=False,  # evita bugs of some builds with past_key_values
        stopping_criteria=stopping,
    )

    # Extraer only the continuation
    new_tokens = gen_ids[:, input_ids.shape[1]:]
    text = tokenizer.decode(new_tokens[0], skip_special_tokens=False)

    # # Stop in the first <|im_end|> if appears
    # if "<|im_end|>" in text:
    #     text = text.split("<|im_end|>", 1)[0]



    return text.strip()

In [ ]:
prompts = [
    {
        "system_msg": (
            "Your role will be that of an assistant that translates Chinese wuxia text into fluent English prose."
            "Your translations should read as if they were written by a native English novelist while preserving the atmosphere and meaning of ancient martial worlds."
            "Avoid literal phrasing and focus on emotional tone and narrative flow. "
            "Always enclose your final English translation between triple square brackets [[[ and ]]] so it can be easily identified."
        ),
        "user_instruction": (
            "Translate the following Chinese text into natural English, making it smooth and stylistically consistent with modern fantasy novels:"
        ),
    },
    {
        "system_msg": (
            "You are an expert literary translator of Chinese wuxia fiction. "
            "Retain the poetic flow, cultural symbolism, and epic atmosphere characteristic of wuxia narratives. "
            "The goal is to produce English reflecting the depth of Chinese idioms. "
            "Surround the complete translated text with [[[ and ]]] to clearly mark the translation."
        ),
        "user_instruction": (
            "Translate the following Chinese passage into refined, literary English that conveys the spirit of classic wuxia storytelling:"
        ),
    },
    {
        "system_msg": (
            "You are functioning as a seq2seq neural translation model trained for Chinese-to-English tasks. "
            "You must produce a one-to-one translation of the source text without commentary, explanation, or rephrasing. "
            "Do not include introductions or stylistic variations, just output the translation. "
            "The translated text must be enclosed between [[[ and ]]] for clear identification."
        ),
        "user_instruction": (
            "Provide a direct English translation of the following Chinese text, maintaining word order and fidelity as much as possible, similar to a seq2seq model:"
        ),
    },
    {
        "system_msg": (
            "You are a bilingual scholar translating Chinese martial arts literature into English. "
            "Your translation must be accurate, faithful, and formal. "
            "Do not include introductions or stylistic variations, just output the translation. "
            "Enclose the entire translation between [[[ and ]]] so it can be clearly extracted."
        ),
        "user_instruction": (
            "Translate the following text into precise English, maintaining original meanings and cultural nuances:"
        ),
    },
    {
        "system_msg": (
            "You are simulating a traditional sequence-to-sequence translation model working on chinese to english translations. "
            "Translate the input text directly into English without adding explanations, literary style, or rewording. "
            "Produce a literal, token-aligned translation suitable for machine translation evaluation. "
            "The output must be wrapped between [[[ and ]]] so it can be parsed automatically."
        ),
        "user_instruction": (
            "Translate the following Chinese text literally into English, preserving structure and meaning with minimal adaptation:"
        ),
    }
]


In [ ]:
import os
import time
from tqdm import tqdm
from datetime import datetime


N = 50
PROMPT_IDS = [0, 1, 2, 3, 4]  # indices of the prompts to evaluar (by default 5 first)
SHOTS_CONFIG = {
    "0shot": 0,
    "1shot": 1,
    "2shot": 2,
    "5shot": 5,
}

output_dir = Path(cfg.translate_dir)
output_dir.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_file = output_dir / f"{cfg.translate_file}_{timestamp}.txt"
time_log_file = output_dir / f"{cfg.translate_file}_{timestamp}_time.txt"

print(f"Saving results to: {output_file}")
print(f"Saving timings to: {time_log_file}")

all_time_records = []




In [ ]:

def build_shots(num_shots, split="train"):
    """Returns a list with num_shots examples of the dataset of training."""
    if num_shots == 0:
        return []
    total_train = len(raw_ds[split])
    return [raw_ds[split][i % total_train] for i in range(num_shots)]


def evaluate_prompt(prompt_id, n=N):
    """Runs the evaluation completes of un prompt (all the configuraciones of shot)."""
    prompt = prompts[prompt_id]
    system_msg = prompt["system_msg"]
    user_instruction = prompt["user_instruction"]

    # Contenedores for results
    log_lines = []
    time_records = []

    log_lines.append(f"\n\n##############################################")
    log_lines.append(f"### PROMPT {prompt_id}")
    log_lines.append(f"### {system_msg[:120]}...")
    log_lines.append(f"##############################################\n")

    prompt_total_time = 0.0 

    for mode, n_shots in SHOTS_CONFIG.items():
        log_lines.append(f"\n=== {mode.upper()} ({n_shots}-shot) ===\n")
        mode_times = []

        for i in tqdm(range(n), desc=f"Prompt {prompt_id} -> {mode}"):
            sample = raw_ds["test"][i % len(raw_ds["test"])]
            shots = build_shots(n_shots)

            start_time = time.perf_counter()
            try:
                translation = translate_with_prompt_qwen(
                    sample["zh"],
                    system_msg=system_msg,
                    user_instruction=user_instruction,
                    shots=shots
                )
            except Exception as e:
                translation = f"[Error in iteration {i}: {e}]"
            elapsed = time.perf_counter() - start_time
            mode_times.append(elapsed)

            # Save in text
            log_lines.append(f"--- Caso {i+1} ---")
            log_lines.append(f"[ZH]: {sample['zh']}")
            log_lines.append(f"[REF]: {sample['en']}")
            log_lines.append(f"[GEN]: {translation.strip()}")
            log_lines.append(f"[TIME]: {elapsed:.2f} s\n")

        avg_time = sum(mode_times) / len(mode_times)
        total_time = sum(mode_times)
        prompt_total_time += total_time
        time_records.append((prompt_id, mode, n_shots, avg_time, total_time))

        log_lines.append(f"\n>>> Time medio {mode}: {avg_time:.2f} s | Total: {total_time:.1f} s\n")

    # summary total by prompt
    log_lines.append(f"\n##### TOTAL PROMPT {prompt_id}: {prompt_total_time/60:.2f} min #####\n")
    return "\n".join(log_lines), time_records


In [ ]:
with open(output_file, "w", encoding="utf-8") as f_out:
    for pid in PROMPT_IDS:
        block_text, time_data = evaluate_prompt(pid, n=N)
        f_out.write(block_text)
        f_out.flush()
        all_time_records.extend(time_data)



In [ ]:

# Summary general
summary = {}
for pid, mode, n_shots, avg_t, total_t in all_time_records:
    summary.setdefault(pid, 0)
    summary[pid] += total_t

print("\n=== SUMMARY TIMES TOTALES (by prompt) ===")
for pid, total in summary.items():
    print(f"Prompt {pid}: {total/60:.2f} min")

# Save summary in the TXT also
with open(time_log_file, "a", encoding="utf-8") as txtfile:
    txtfile.write("\n=== SUMMARY TIMES TOTALES (by prompt) ===\n")
    for pid, total in summary.items():
        txtfile.write(f"Prompt {pid}: {total/60:.2f} min\n")

print(f"\nEvaluation completed.\nResults: {output_file}\nTimings: {time_log_file}")
